# Part 4: Model Training and Evaluation
This notebook builds the machine learning models, benchmarks estimators, analyzes feature importances, and saves the production pipeline model.


In [ ]:
import pandas as pd
import numpy as np
import joblib
from sklearn.model_selection import (
train_test_split,
StratifiedKFold,
cross_validate,
GridSearchCV
)

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer

from sklearn.preprocessing import (
RobustScaler,
OneHotEncoder
)

from sklearn.linear_model import LogisticRegression

from sklearn.tree import DecisionTreeClassifier

from sklearn.ensemble import (
RandomForestClassifier,
GradientBoostingClassifier
)

from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB

from xgboost import XGBClassifier

from imblearn.over_sampling import SMOTE
from imblearn.pipeline import Pipeline as ImbPipeline

from sklearn.metrics import (
accuracy_score,
precision_score,
recall_score,
f1_score,
roc_auc_score,
classification_report,
confusion_matrix
)
import matplotlib.pyplot as plt
import seaborn as sns
df = pd.read_csv('../data/processed_telco_churn.csv')



# SEPARATE FEATURES AND TARGET

In [117]:
X = df.drop(
"Churn",
axis=1
)

y = df["Churn"]

# TRAIN-TEST SPLIT

In [118]:
X_train, X_test, y_train, y_test = train_test_split(
X,
y,
test_size=0.20,
random_state=42,
stratify=y
)

# DEFINE NUMERICAL FEATURES

In [119]:
numeric_features = [
"SeniorCitizen",
"tenure",
"MonthlyCharges",
"TotalCharges",
"AverageMonthlySpend"
]

# DEFINE CATEGORICAL FEATURES

In [120]:

categorical_features = [
col
for col in X.columns
if col not in numeric_features
]



# NUMERICAL PREPROCESSING

In [121]:
numeric_pipeline = Pipeline(
steps=[
(
"imputer",
SimpleImputer(strategy="median")
),

    (
        "scaler",
        RobustScaler()
    )
]

)

# CATEGORICAL PREPROCESSING

In [122]:
categorical_pipeline = Pipeline(
steps=[
(
"imputer",
SimpleImputer(strategy="most_frequent")
),

    (
        "encoder",
        OneHotEncoder(
            handle_unknown="ignore",
            drop="first",
            sparse_output=False
        )
    )
]

)

# COLUMN TRANSFORMER

In [123]:
preprocessor = ColumnTransformer(
transformers=[
(
"numerical",
numeric_pipeline,
numeric_features
),

    (
        "categorical",
        categorical_pipeline,
        categorical_features
    )
]

)

# CREATE MODEL PIPELINES

In [124]:
pipelines = {

"Logistic Regression": ImbPipeline(
    steps=[
        (
            "preprocessor",
            preprocessor
        ),

        (
            "smote",
            SMOTE(
                random_state=42
            )
        ),

        (
            "model",
            LogisticRegression(
                max_iter=1000,
                random_state=42
            )
        )
    ]
),


"Decision Tree": ImbPipeline(
    steps=[
        (
            "preprocessor",
            preprocessor
        ),

        (
            "smote",
            SMOTE(
                random_state=42
            )
        ),

        (
            "model",
            DecisionTreeClassifier(
                random_state=42
            )
        )
    ]
),


"Random Forest": ImbPipeline(
    steps=[
        (
            "preprocessor",
            preprocessor
        ),

        (
            "smote",
            SMOTE(
                random_state=42
            )
        ),

        (
            "model",
            RandomForestClassifier(
                random_state=42
            )
        )
    ]
),


"Gradient Boosting": ImbPipeline(
    steps=[
        (
            "preprocessor",
            preprocessor
        ),

        (
            "smote",
            SMOTE(
                random_state=42
            )
        ),

        (
            "model",
            GradientBoostingClassifier(
                random_state=42
            )
        )
    ]
),


"Support Vector Machine": ImbPipeline(
    steps=[
        (
            "preprocessor",
            preprocessor
        ),

        (
            "smote",
            SMOTE(
                random_state=42
            )
        ),

        (
            "model",
            SVC(
                probability=True,
                random_state=42
            )
        )
    ]
),


"K-Nearest Neighbors": ImbPipeline(
    steps=[
        (
            "preprocessor",
            preprocessor
        ),

        (
            "smote",
            SMOTE(
                random_state=42
            )
        ),

        (
            "model",
            KNeighborsClassifier()
        )
    ]
),


"Gaussian Naive Bayes": ImbPipeline(
    steps=[
        (
            "preprocessor",
            preprocessor
        ),

        (
            "smote",
            SMOTE(
                random_state=42
            )
        ),

        (
            "model",
            GaussianNB()
        )
    ]
),


"XGBoost": ImbPipeline(
    steps=[
        (
            "preprocessor",
            preprocessor
        ),

        (
            "smote",
            SMOTE(
                random_state=42
            )
        ),

        (
            "model",
            XGBClassifier(
                random_state=42,
                eval_metric="logloss"
            )
        )
    ]
)

}

# STRATIFIED K-FOLD

In [125]:
skf = StratifiedKFold(
n_splits=5,
shuffle=True,
random_state=42
)

 # MODEL COMPARISON

In [126]:
results = []

for name, model in pipelines.items():


    scores = cross_validate(
    model,
    X_train,
    y_train,
    cv=skf,
    scoring=[
        "accuracy",
        "precision",
        "recall",
        "f1",
        "roc_auc"
    ],
    n_jobs=-1
)

    results.append({

    "Model": name,

    "Accuracy":
        scores["test_accuracy"].mean(),

    "Precision":
        scores["test_precision"].mean(),

    "Recall":
        scores["test_recall"].mean(),

    "F1 Score":
        scores["test_f1"].mean(),

    "ROC AUC":
        scores["test_roc_auc"].mean()
})

In [127]:
results_df = pd.DataFrame(
results
)

results_df = results_df.sort_values(
by="F1 Score",
ascending=False
)

print(results_df)

                    Model  Accuracy  Precision    Recall  F1 Score   ROC AUC
3       Gradient Boosting  0.787542   0.583996  0.694314  0.634299  0.846237
0     Logistic Regression  0.758968   0.531394  0.783278  0.633004  0.847488
4  Support Vector Machine  0.767128   0.545964  0.725753  0.622963  0.828619
2           Random Forest  0.785234   0.593042  0.607358  0.600030  0.824366
6    Gaussian Naive Bayes  0.668973   0.438028  0.871572  0.582938  0.819691
7                 XGBoost  0.778135   0.583955  0.571906  0.577714  0.817469
5     K-Nearest Neighbors  0.696308   0.455012  0.729766  0.560428  0.761171
1           Decision Tree  0.725775   0.486023  0.559197  0.519896  0.673090


# Hyperparameter Tuning

In [128]:
lr_pipeline = pipelines["Logistic Regression"]

param_grid = {
    "model__C": [0.01, 0.1, 1, 5, 10],
    "model__penalty": ["l1", "l2"],
    "model__solver": ["liblinear", "saga"],
    "model__class_weight": [None, "balanced"]
}


grid = GridSearchCV(
    estimator=lr_pipeline,
    param_grid=param_grid,
    cv=skf,
    scoring="f1",
    n_jobs=-1,
    verbose=1
)

grid.fit(X_train, y_train)

print("Best Parameters:")
print(grid.best_params_)

print("\nBest Cross-Validation F1 Score:")
print(grid.best_score_)

# Best Pipeline
best_lr = grid.best_estimator_


Fitting 5 folds for each of 40 candidates, totalling 200 fits


c:\Users\shubh\anaconda3\Lib\site-packages\sklearn\linear_model\_logistic.py:1135: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', and C=np.inf instead of penalty=None.
  warnings.warn(
c:\Users\shubh\anaconda3\Lib\site-packages\sklearn\linear_model\_logistic.py:1160: UserWarning: Inconsistent values: penalty=l1 with l1_ratio=0.0. penalty is deprecated. Please use l1_ratio only.
  warnings.warn(


Best Parameters:
{'model__C': 0.1, 'model__class_weight': None, 'model__penalty': 'l1', 'model__solver': 'saga'}

Best Cross-Validation F1 Score:
0.6348914050966231


# Predictions

In [129]:
y_pred = best_lr.predict(X_test)
y_prob = best_lr.predict_proba(X_test)[:, 1]

In [130]:
print("Accuracy :", accuracy_score(y_test, y_pred))
print("Precision:", precision_score(y_test, y_pred))
print("Recall :", recall_score(y_test, y_pred))
print("F1 Score :", f1_score(y_test, y_pred))
print("ROC-AUC :", roc_auc_score(y_test, y_prob))

print("\nClassification Report")
print(classification_report(y_test, y_pred))

print("\nConfusion Matrix")
print(confusion_matrix(y_test, y_pred))

Accuracy : 0.7430801987224982
Precision: 0.5104166666666666
Recall : 0.786096256684492
F1 Score : 0.6189473684210526
ROC-AUC : 0.8444392776873595

Classification Report
              precision    recall  f1-score   support

           0       0.90      0.73      0.81      1035
           1       0.51      0.79      0.62       374

    accuracy                           0.74      1409
   macro avg       0.71      0.76      0.71      1409
weighted avg       0.80      0.74      0.76      1409


Confusion Matrix
[[753 282]
 [ 80 294]]


 # Baseline Model (Logistic Regression) Confusion Matrix

In [1]:
cm_lr = confusion_matrix(y_test, y_pred)
sns.heatmap(cm_lr,annot=True,fmt='d')
plt.title('Baseline Model(Logistic Regression) Confusion Matrix')
plt.ylabel('Actual')
plt.xlabel('Predicted')

plt.savefig(
    "../plots/Model Evaluation/logistic regression confusion_matrix.png",
    dpi=300,
    bbox_inches="tight"
)
plt.show()


NameError: name 'confusion_matrix' is not defined

# Save the Production Model

In [132]:
joblib.dump(best_lr, '..\models\customer_churn_model.pkl')
print("Production model successfully saved to 'customer_churn_model.pkl'")

Production model successfully saved to 'customer_churn_model.pkl'
